In [ ]:
%py
# PySpark code to mask the last 4 digits of invoice numbers in d_product_revenue_clone table

from pyspark.sql.functions import when, lit, length, substring

# Sample schema (replace with actual schema of d_product_revenue)
from pyspark.sql.types import StructType, StructField, StringType, LongType

schema = StructType([
    StructField("invoice_number", StringType(), True), # Assuming invoice_number can be null
    StructField("other_column", StringType(), True),  # Add other columns as needed
    # ...
])


data = [
    ("1234567890", "some value"),  # Happy Path - String > 4 chars
    ("1234567", "other value"),  # Edge Case - String == length 4 chars
    ("123", "another value"),    # Edge Case - String < 4 chars
    (None, "null invoice"),        # NULL handling
    ("", "empty invoice"),      # Empty String handling
    ("ABC1234567", "alphanumeric"), # Non-numeric (handle according to requirement)

    ("9876543210", "some value 2"),  # Happy path - String > 4 chars
    ("654321", "other value 2"),  # Edge Case - String == length 4 chars
    ("456", "another value 2"),    # Edge Case - String < 4 chars
    (None, "null invoice 2"),        # NULL handling
    ("", "empty invoice 2"),      # Empty String handling
    ("DEF7654321", "alphanumeric 2"), # Non-numeric

    ("1111111111", "some value 3"),  # Happy path - String > 4 chars
    ("9999999", "other value 3"),  # Edge Case - String == length 4 chars
    ("888", "another value 3"),    # Edge Case - String < 4 chars
    (None, "null invoice 3"),        # NULL handling
    ("", "empty invoice 3"),      # Empty String handling
    ("GHI2345678", "alphanumeric 3"), # Non-numeric

    ("1234567890123456", "long invoice"),  # Very Long invoice number
    ("12", "very short invoice"),        # Very Short Invoice number

    ("XYZ123", "alphanumeric short"), # Non-numeric, short invoice
    ("PQRS12345678", "alphanumeric long") # Non-numeric, long invoice
]




df = spark.createDataFrame(data, schema=schema)

# Masking logic
df_masked = df.withColumn("masked_invoice", when(
    df["invoice_number"].isNull() | (df["invoice_number"] == ""), # Handle NULLs and empty strings
    df["invoice_number"]
).otherwise(when(
    length(df["invoice_number"]) <= 4,
    lit("****")                                                 # Mask fully if <= 4 chars
).otherwise(
    substring(df["invoice_number"], 1, length(df["invoice_number"]) - 4) + lit("****") # Mask last 4
)))



# Display the masked data (remove in production)
df_masked.display()


